In [3]:
import teehr
import pandas as pd
from teehr.evaluation.spark_session_utils import create_spark_session

from teehr import DeterministicMetrics as dm
from teehr import Signatures as s
from teehr import RowLevelCalculatedFields as rcf
from teehr import TimeseriesAwareCalculatedFields as tcf
from teehr import Bootstrappers as bs

from teehr.models.filters import TableFilter

from pyspark.sql import functions as F

from pyspark.sql import DataFrame

import copy
import time

teehr.__version__

'0.6.6'

In [4]:
import os

# Alternate executor pod template targeting the ON-DEMAND `nb-r5-4xlarge-teehr`
# node group instead of the spot `spark-r5-4xlarge-spot` pool, for tuning runs
# where we want clean measurements without spot-interruption noise. Same
# instance type (r5.4xlarge) so executor sizing math stays comparable to prior
# spot-based runs. Different taint on this node group (hub.jupyter.org/dedicated
# =user vs teehr-hub/dedicated=worker), so it needs its own tolerations.
ONDEMAND_POD_TEMPLATE_PATH = os.path.expanduser("~/executor-pod-template-ondemand.yaml")

with open(ONDEMAND_POD_TEMPLATE_PATH, "w") as f:
    f.write("""apiVersion: v1
kind: Pod
spec:
  terminationGracePeriodSeconds: 60
  securityContext:
    runAsUser: 1000
    runAsGroup: 1000
    fsGroup: 1000
  containers:
  - name: spark-kubernetes-executor
    securityContext:
      runAsUser: 1000
      runAsGroup: 1000
      allowPrivilegeEscalation: false
    lifecycle:
      preStop:
        exec:
          command: ["/bin/sh", "-c", "sleep 30"]
    volumeMounts:
    - name: data-nfs
      mountPath: /data
  volumes:
  - name: data-nfs
    persistentVolumeClaim:
      claimName: data-nfs
  tolerations:
  - effect: "NoSchedule"
    key: "hub.jupyter.org/dedicated"
    operator: "Equal"
    value: "user"
  - effect: "NoSchedule"
    key: "hub.jupyter.org_dedicated"
    operator: "Equal"
    value: "user"
  nodeSelector:
    teehr-hub/nodegroup-name: nb-r5-4xlarge
""")

print(f"Wrote alternate pod template to {ONDEMAND_POD_TEMPLATE_PATH}")


Wrote alternate pod template to /home/jovyan/executor-pod-template-ondemand.yaml


In [8]:
def unpack_quantile_bootstrap_columns(table, metrics):
    sdf = table.to_sdf()
    for m in metrics:
        if not getattr(m, "bootstrap", None):
            continue
        for q in m.bootstrap.quantiles:
            key = f"{m.output_field_name}_{q}"
            sdf = sdf.withColumn(key.replace(".", "_"), F.col(m.output_field_name).getItem(key))
        sdf = sdf.drop(m.output_field_name)
    return table._with_sdf(sdf)

# Validation: legacy vs. vectorized bootstrap engine

Runs teehr's real `.aggregate()` (through actual Spark pandas_udf execution on
executors, not the in-process unit tests in the teehr repo) against a small
synthetic dataset, once per engine, and compares the bootstrap quantile
outputs. This specifically validates that `spark.executorEnv.TEEHR_BOOTSTRAP_ENGINE`
actually propagates to executors in *this deployed image* and that the real
Spark/pandas_udf path produces equivalent results -- the teehr repo's own
test suite (`tests/query/test_vectorized_bootstrap_funcs.py`) already
validates the underlying math in-process; this validates it end-to-end.

Uses synthetic data (not the real warehouse tables) so this runs in seconds
and isolates the engine-comparison question from anything about the upstream
pipeline, which has already been validated separately.


In [5]:
# Small, fast cluster -- this is a correctness spot-check, not a performance
# test, so it doesn't need production-scale resources.
def run_engine_check(engine, n_groups=6, n_per_group=40, reps=1000):
    """Run the real Spark aggregate() with bootstrap metrics on synthetic data.

    engine: "legacy" (default teehr behavior) or "vectorized" (opt-in via
    spark.executorEnv.TEEHR_BOOTSTRAP_ENGINE, which is how the flag actually
    reaches the pandas_udf running on executors -- setting os.environ in the
    notebook itself only affects the driver process, not executors).
    """
    update_configs = {
        "spark.sql.shuffle.partitions": 64,
        "spark.sql.adaptive.coalescePartitions.enabled": "false",
    }
    if engine == "vectorized":
        update_configs["spark.executorEnv.TEEHR_BOOTSTRAP_ENGINE"] = "vectorized"

    chk_spark = create_spark_session(
        start_spark_cluster=True,
        executor_instances=4,
        executor_memory="8g",
        executor_cores=2,
        aws_profile="default",
        pod_template_path=ONDEMAND_POD_TEMPLATE_PATH,
        update_configs=update_configs,
    )
    try:
        rng = np.random.default_rng(0)
        rows = []
        for g in range(n_groups):
            for _ in range(n_per_group):
                rows.append({
                    "group_id": g,
                    "primary_value": float(abs(rng.normal(10, 3))) + 0.1,
                    "secondary_value": float(abs(rng.normal(9, 4))) + 0.1,
                })
        # .aggregate() is only available on teehr's own wrapper objects, not on a
        # plain Spark DataFrame -- grab any small real table just to get a valid
        # wrapper instance, then swap in our synthetic data via the same
        # ._with_sdf() mechanism used by unpack_quantile_bootstrap_columns above.
        chk_ev = teehr.RemoteReadWriteEvaluation(spark=chk_spark, enable_spark_proxy=True)
        base_table = chk_ev.table("locations")
        sdf_in = chk_spark.createDataFrame(pd.DataFrame(rows))
        chk_table = base_table._with_sdf(sdf_in)

        chk_bootstrap = bs.Stationary(reps=reps, seed=1234, quantiles=[0.025, 0.975])
        chk_metrics = [
            dm.RelativeMean(output_field_name="relative_mean_boot", bootstrap=chk_bootstrap),
            dm.NashSutcliffeEfficiency(output_field_name="nse_boot", bootstrap=chk_bootstrap),
            dm.KlingGuptaEfficiency(output_field_name="kge_boot", bootstrap=chk_bootstrap),
            dm.PearsonCorrelation(output_field_name="pearson_boot", bootstrap=chk_bootstrap),
        ]

        chk_results = chk_table.aggregate(group_by=["group_id"], metrics=chk_metrics)
        chk_results = unpack_quantile_bootstrap_columns(chk_results, chk_metrics)
        pdf = chk_results.to_sdf().orderBy("group_id").toPandas()
        return pdf
    finally:
        chk_spark.stop()


In [6]:
import numpy as np

In [9]:
def compare_engine_runs(df_a, df_b, label_a, label_b, tolerance=1e-4):
    """Compare two run_engine_check() outputs and print a max-diff summary.

    Not a strict pass/fail on its own -- see the control-test cell below,
    which uses this to check whether legacy-vs-legacy (same code, two
    separate sessions) already shows similar-sized differences due to
    Spark's row-order variability across runs, before blaming the engine.
    """
    compare_cols = [c for c in df_a.columns if c != "group_id"]
    merged = df_a.merge(df_b, on="group_id", suffixes=(f"_{label_a}", f"_{label_b}"))

    print(f"{'column':35s} {'max_abs_diff':>14s} {'max_rel_diff':>14s}")
    worst_rel = 0.0
    for col in compare_cols:
        a = merged[f"{col}_{label_a}"].to_numpy(dtype=float)
        b = merged[f"{col}_{label_b}"].to_numpy(dtype=float)
        abs_diff = np.abs(a - b)
        rel_diff = abs_diff / np.maximum(np.abs(a), 1e-12)
        worst_rel = max(worst_rel, rel_diff.max())
        print(f"{col:35s} {abs_diff.max():14.3e} {rel_diff.max():14.3e}")

    if worst_rel < tolerance:
        print(f"\nmax relative diff {worst_rel:.2e} < tolerance {tolerance:.0e}.")
    else:
        print(f"\nmax relative diff {worst_rel:.2e} EXCEEDS tolerance {tolerance:.0e}.")
    return merged, worst_rel


legacy_df = run_engine_check("legacy")
vectorized_df = run_engine_check("vectorized")

print("=== legacy vs. vectorized ===")
merged, worst_rel_engine = compare_engine_runs(legacy_df, vectorized_df, "legacy", "vectorized")
merged


INFO:teehr.evaluation.spark_session_utils:🚀 Creating Spark session: TEEHR Evaluation
INFO:teehr.evaluation.spark_session_utils:📦 Configuring Spark cluster with container image: None
INFO:teehr.evaluation.spark_session_utils:🔍 Initial spark namespace from ENV: teehr-hub
INFO:teehr.evaluation.spark_session_utils:🔍 Connecting to Kubernetes API: https://172.20.0.1:443
INFO:teehr.evaluation.spark_session_utils:🎯 Executor namespace: teehr-hub
INFO:teehr.evaluation.spark_session_utils:🔐 Executor service account: spark (in teehr-hub)
INFO:teehr.evaluation.spark_session_utils:🔐 Using in-cluster authentication
INFO:teehr.evaluation.spark_session_utils:🔗 Setting driver host to pod IP: 10.0.3.91
INFO:teehr.evaluation.spark_session_utils:✅ Spark cluster configuration successful!
INFO:teehr.evaluation.spark_session_utils:   - Executor instances: 4
INFO:teehr.evaluation.spark_session_utils:   - Executor memory: 8g
INFO:teehr.evaluation.spark_session_utils:   - Executor cores: 2
INFO:teehr.evaluation.

=== legacy vs. vectorized ===
column                                max_abs_diff   max_rel_diff
relative_mean_boot_0_025                 0.000e+00      0.000e+00
relative_mean_boot_0_975                 0.000e+00      0.000e+00
nse_boot_0_025                           0.000e+00      0.000e+00
nse_boot_0_975                           0.000e+00      0.000e+00
kge_boot_0_025                           0.000e+00      0.000e+00
kge_boot_0_975                           0.000e+00      0.000e+00
pearson_boot_0_025                       0.000e+00      0.000e+00
pearson_boot_0_975                       0.000e+00      0.000e+00

max relative diff 0.00e+00 < tolerance 1e-04.


,group_id,relative_mean_boot_0_025_legacy,relative_mean_boot_0_975_legacy,nse_boot_0_025_legacy,nse_boot_0_975_legacy,kge_boot_0_025_legacy,kge_boot_0_975_legacy,pearson_boot_0_025_legacy,pearson_boot_0_975_legacy,relative_mean_boot_0_025_vectorized,relative_mean_boot_0_975_vectorized,nse_boot_0_025_vectorized,nse_boot_0_975_vectorized,kge_boot_0_025_vectorized,kge_boot_0_975_vectorized,pearson_boot_0_025_vectorized,pearson_boot_0_975_vectorized
0,0,0.824963,1.088090,-2.981449,-0.154552,-0.270573,0.480302,-0.168473,0.501865,0.824963,1.088090,-2.981449,-0.154552,-0.270573,0.480302,-0.168473,0.501865
1,1,0.854046,1.174456,-3.599711,-0.971939,-0.340438,0.199428,-0.169909,0.370520,0.854046,1.174456,-3.599711,-0.971939,-0.340438,0.199428,-0.169909,0.370520
2,2,0.851192,1.162935,-4.948938,-0.311604,-0.570123,0.404245,-0.164816,0.437113,0.851192,1.162935,-4.948938,-0.311604,-0.570123,0.404245,-0.164816,0.437113
3,3,0.819276,1.113215,-3.160092,-0.329550,-0.353832,0.302563,-0.286196,0.321404,0.819276,1.113215,-3.160092,-0.329550,-0.353832,0.302563,-0.286196,0.321404
4,4,0.699485,0.979389,-4.665993,-1.207668,-0.476821,0.179482,-0.417415,0.270398,0.699485,0.979389,-4.665993,-1.207668,-0.476821,0.179482,-0.417415,0.270398
5,5,0.915431,1.308737,-3.063011,-0.904277,-0.505773,0.177906,-0.490596,0.241664,0.915431,1.308737,-3.063011,-0.904277,-0.505773,0.177906,-0.490596,0.241664


## Control test: legacy vs. legacy (same code, two separate sessions)

If this shows differences of a similar size to the legacy-vs-vectorized
comparison above, that confirms the cause is Spark's row-order variability
across separate job runs (bootstrap resampling is position-based, and
grouped-aggregate pandas_udf doesn't guarantee identical row order across
runs) -- not a bug in the vectorized engine. The teehr repo's own unit tests
already confirmed the vectorized math matches the legacy math exactly when
row order is controlled (same in-memory pandas Series, no Spark shuffle).


In [ ]:
legacy_df_a = run_engine_check("legacy")
legacy_df_b = run_engine_check("legacy")

print("=== legacy (run A) vs. legacy (run B) -- control ===")
merged_control, worst_rel_control = compare_engine_runs(legacy_df_a, legacy_df_b, "a", "b")

print(f"\nlegacy-vs-vectorized worst relative diff: {worst_rel_engine:.2e}")
print(f"legacy-vs-legacy (control) worst relative diff: {worst_rel_control:.2e}")
if worst_rel_control > worst_rel_engine / 3:
    print("\nControl shows similarly-sized differences -- consistent with row-order")
    print("variability across separate Spark sessions, not an engine bug.")
else:
    print("\nControl is much tighter than the engine comparison -- worth investigating")
    print("further, since row-order variability alone may not explain the gap.")


In [ ]:
spark.stop()

NameError: name 'spark' is not defined